# FairMALLM-GAN v4 — ablation without D2 (no_fair)

v4 = v3 + activity plans for quiz/assessment weeks (see the docstring of fair_mallm_gan_v4.py).

**Settings:** Accelerator **GPU T4 x2**, Internet **on**, input **oulad-processed**.

Run with *Save Version → Save & Run All (Commit)*. Results: Output tab, `run_*/` and `eval_*/`.

In [ ]:
!pip install -q transformers accelerate


In [ ]:
%%writefile common.py
# -*- coding: utf-8 -*-
"""
common.py  --  shared helpers for FairMALLM-GAN v3
===================================================
Used by: prepare_cohort.py, fair_mallm_gan_v3.py, baseline_ctgan.py, evaluate_v3.py

Contents
  * constants (feature order, short tags)
  * load_real()          : loads the processed OULAD tensors (unchanged files)
  * build_cohort_splits(): cohort filter + grouped, stratified 70/15/15 split
  * summary features     : 31-dim temporal summary vector per student (D1/D2)
  * fairness metrics     : SPD, EOD
"""
import os, json
import numpy as np
import pandas as pd

FEATURES = ["homepage_clicks", "subpage_clicks", "oucontent_clicks",
            "quiz_interactions", "weekly_assessment_score"]
TAGS = ["hp", "sp", "oc", "qz", "sc"]
SCORE_IDX = 4                     # weekly_assessment_score is bounded 0..100
T_WEEKS = 12


# =============================================================================
# 1. LOAD THE PROCESSED OULAD FILES (produced by Weekly_binning.ipynb)
# =============================================================================
def load_real(data_dir):
    p = lambda f: os.path.join(data_dir, f)
    d = dict(X=np.load(p("X_seq.npy")).astype(np.float32),
             mask=np.load(p("mask.npy")).astype(np.int8),
             y=np.load(p("y.npy")).astype(np.int8),
             S=np.load(p("S_gender.npy")).astype(np.int8),
             keys=np.load(p("student_keys.npy"), allow_pickle=True))
    # '?' = missing in OULAD -> must be parsed as NaN (v2 notebook did not do this)
    st = pd.read_csv(p("student_static.csv"), na_values="?", low_memory=False)
    assert (st["student_key"].values == d["keys"]).all(), "static table not aligned"
    d["static"] = st
    return d


# =============================================================================
# 2. COHORT + GROUPED STRATIFIED SPLIT
# =============================================================================
def active_at_cutoff(static):
    """True if the student was still registered at the early-warning cutoff.
    Students who unregistered before the cutoff (incl. before day 0) already
    have a known outcome at prediction time -> not an early-warning case."""
    unreg = pd.to_numeric(static["date_unregistration"], errors="coerce")
    return ~(unreg.notna() & (unreg < static["cutoff_days"])).values


def build_cohort_splits(d, seed=42):
    """Grouped (by id_student) split, stratified by label x gender, 70/15/15.
    The same person never appears in two splits (v2 split leaked 15%).
    Returns dict with index arrays for cohort 'all' and 'active'."""
    from sklearn.model_selection import StratifiedGroupKFold
    st = d["static"]
    strata = d["y"].astype(int) * 2 + d["S"].astype(int)
    groups = st["id_student"].values
    sgk = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=seed)
    fold = np.empty(len(strata), int)
    for k, (_, te) in enumerate(sgk.split(np.zeros(len(strata)), strata, groups)):
        fold[te] = k
    split = np.where(fold < 14, "train", np.where(fold < 17, "val", "test"))
    act = active_at_cutoff(st)
    out = {"split": split, "active": act}
    for coh, keep in [("all", np.ones(len(act), bool)), ("active", act)]:
        for s in ["train", "val", "test"]:
            out[f"{coh}_{s}"] = np.where((split == s) & keep)[0]
    return out


def get_split(d, cohort="active", seed=42):
    sp = build_cohort_splits(d, seed)
    return sp[f"{cohort}_train"], sp[f"{cohort}_val"], sp[f"{cohort}_test"]


# =============================================================================
# 3. TEMPORAL SUMMARY FEATURES FOR D1 / D2  (log scale, 31 dims)
# =============================================================================
STATS = ["total", "weekly mean", "last week", "trend", "share of active weeks"]


def build_matrix(X, M):
    """Vectorised: (N,T,F) counts + (N,T) mask -> (N, 5*F+1) summary matrix.
    Per feature (log1p scale): total, weekly mean, last real week,
    least-squares trend over the real weeks, share of weeks with activity.
    Assumes real weeks are contiguous from week 0 (true for OULAD + v3 output)."""
    X = np.clip(np.asarray(X, np.float64), 0, None)
    M = np.asarray(M, np.float64)
    N, T, F = X.shape
    L = np.maximum(M.sum(1), 1)                               # (N,)
    V = np.log1p(X) * M[:, :, None]                           # (N,T,F)
    tot = V.sum(1)
    mean = tot / L[:, None]
    last = V[np.arange(N), (L - 1).astype(int)]               # (N,F)
    t = np.arange(T)[None, :] * M                             # (N,T)
    tbar = t.sum(1) / L
    dt = (np.arange(T)[None, :] - tbar[:, None]) * M          # (N,T)
    den = np.maximum((dt ** 2).sum(1), 1e-9)
    slope = (dt[:, :, None] * (V - mean[:, None, :] * M[:, :, None])).sum(1) / den[:, None]
    active = ((X > 0) * M[:, :, None]).sum(1) / L[:, None]
    out = np.stack([tot, mean, last, slope, active], axis=2).reshape(N, F * 5)
    return np.column_stack([out, M.sum(1)]).astype(np.float32)


def to_vec(x, m):
    return build_matrix(x[None], m[None])[0]


def raw_stat(X, M, f, stat):
    """The same statistics on the ORIGINAL scale (for human-readable feedback)."""
    X = np.clip(np.asarray(X, np.float64), 0, None); M = np.asarray(M, np.float64)
    L = np.maximum(M.sum(1), 1); v = X[:, :, f] * M
    if stat == "total":        return v.sum(1)
    if stat == "weekly mean":  return v.sum(1) / L
    if stat == "last week":    return v[np.arange(len(v)), (L - 1).astype(int)]
    if stat == "share of active weeks": return ((v > 0) * M).sum(1) / L
    if stat == "trend":
        return (v[np.arange(len(v)), (L - 1).astype(int)] - v[:, 0]) / np.maximum(L - 1, 1)
    raise ValueError(stat)


def vec_name(i):
    if i >= 5 * len(STATS):
        return "sequence length"
    return f"{FEATURES[i // len(STATS)]} ({STATS[i % len(STATS)]})"


# =============================================================================
# 4. FAIRNESS METRICS   (S=1 female, S=0 male)  -- same definitions as proposal
# =============================================================================
def spd(y_pred, S):
    """SPD = P(yhat=1 | S=0) - P(yhat=1 | S=1)"""
    y_pred, S = np.asarray(y_pred), np.asarray(S)
    return float(y_pred[S == 0].mean() - y_pred[S == 1].mean())


def eod(y_true, y_pred, S):
    """EOD = P(yhat=1 | y=1, S=0) - P(yhat=1 | y=1, S=1)"""
    y_true, y_pred, S = map(np.asarray, (y_true, y_pred, S))
    a = y_pred[(y_true == 1) & (S == 0)].mean()
    b = y_pred[(y_true == 1) & (S == 1)].mean()
    return float(a - b)


# =============================================================================
# 5. SMALL UTILITIES
# =============================================================================
def flat_log(X, M):
    """(N,T,F) -> (N,T*F) log1p with padded weeks = 0 (used for privacy DCR)."""
    Z = np.log1p(np.clip(X, 0, None)) * M[:, :, None]
    return Z.reshape(len(Z), -1)


def save_json(obj, path):
    def conv(o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        raise TypeError(type(o))
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2, default=conv)


In [ ]:
%%writefile fair_mallm_gan_v4.py
# -*- coding: utf-8 -*-
"""
FairMALLM-GAN v4 : Fair & Temporally-Aware LLM-Guided Synthetic Data Generation
================================================================================
v4 = v3 + realism changes for quiz / assessment weeks (Oct 2026). Everything
else (LLM, discriminators, filters, modes, evaluation) is unchanged.

WHAT CHANGED vs v3
  A. Activity plan: every requested student comes with the quiz (qz) and
     assessment (sc) weeks of a randomly drawn REAL training student with the
     same outcome Y (mode full: drawn regardless of gender, so the plan carries
     no gender signal; modes no_fair: same S x Y cell, i.e. real mix). The LLM
     still writes every number. v3 produced qz>0 in 7% / sc>0 in 6% of weeks
     against 19% / 19% in real data.
  B. Students that fill less than plan_min_recall of their planned qz/sc weeks
     are rejected ("plan_mismatch").
  C. Few-shot examples show their own plan; active_shot_frac of them have
     quiz/assessment activity.
  D. Feedback always reports qz/sc active-week rates (real vs synthetic) and
     plan compliance; data card adds % of weeks with quiz by week.
  E. zero_cap_margin 0.03 -> 0.01 (v3: 5.2% all-zero students vs 2.3% real).
  The flat MALLM-GAN baseline (mode mallm_flat) never uses A-C.
  v4.1 (after the 21 Oct smoke test: plan recall 0.52, Qwen swapped the qz and
  sc columns and echoed the plan): the plan is given as a fill-in TEMPLATE per
  student, e.g. <S=1;Y=0;W=12> w0:_,_,_,0,0 | w1:_,_,_,Q,S | ..., so the model
  only replaces _ / Q / S by numbers; few-shot examples are shown without plans.
  v4.2 (after smoke test 2: template recall 0.49, sp/oc collapsed to 0, OOM):
  plan_mode="constrained" (default). The prompt is the plain v3 prompt (headers
  only); the plan is enforced while decoding by a logits processor
  (PlanConstraint): in the qz field the first digit must be 1-9 in planned
  weeks and the value must be 0 in unplanned weeks; the same for sc. hp, sp,
  oc and all actual values stay the LLM's choice (grammar-constrained decoding).
  v4.3 (after smoke test 3: recall 1.0, qz 19.5% / sc 18.6% active weeks, but Qwen
  wrote the smallest allowed value, mostly qz=1): the plan also carries the NUMBER
  OF DIGITS of the donor's qz / sc value in each planned week (order of magnitude
  only); the decoder enforces that length, the LLM still chooses the digits.

(v3 notes below)
One frozen generator LLM + two logistic-regression discriminators
  D1 = realism  (real vs synthetic weekly sequences)   -> target accuracy 0.5
  D2 = fairness (predict gender S from features + Y)   -> target AUC      0.5
Discriminator results are turned into natural-language feedback and put into
the next generation prompt (prompt optimisation, no fine-tuning).

WHAT CHANGED vs v2 (details in CHANGES_v2_to_v3.md)
  1. Conditioned slots: the prompt lists the exact headers <S=..;Y=..;W=..> to
     fill, so the gender x label mix is controlled (v2 let the LLM choose and
     the synthetic data ended up with SPD -0.20 although real SPD is +0.02).
  2. Compact serialisation  w0:5,2,10,0,61 | w1:...  (~40% fewer tokens) and
     a strict parser (wrong number of weeks -> rejected; v2 accepted 8 weeks).
  3. Batched generation: many prompts in one GPU pass (v2: one prompt at a
     time, ~7 min per 8 students).
  4. Few-shot examples stratified by S x Y; aggregate "data card" of the real
     training data in the system prompt.
  5. D1/D2 on log-scaled temporal summaries; D1 compares against a real
     sample with the same S x Y mix; specific, numeric feedback (which
     statistic differs, real vs synthetic) instead of one generic sentence.
  6. Rejection filters: near-copies of real training students, duplicates and
     gender-swapped twins of accepted synthetic students, impossible lengths,
     too many all-zero students.
  7. Three modes for RQ3:  full | no_fair | mallm_flat (see Config.mode)
  8. Time budget -> clean stop before Kaggle's 12 h limit; checkpoint/resume.
"""
import os, re, json, time, math, warnings
from dataclasses import dataclass, asdict
from typing import List, Optional

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.neighbors import NearestNeighbors

from common import (FEATURES, TAGS, SCORE_IDX, T_WEEKS, STATS, load_real,
                    get_split, build_matrix, vec_name, raw_stat, spd,
                    flat_log, save_json)

warnings.filterwarnings("ignore")


# =============================================================================
# CONFIG
# =============================================================================
@dataclass
class Config:
    DATA_DIR: str = "/kaggle/input/oulad-processed/oulad_processed"
    OUT_DIR: str = "/kaggle/working/run_full"

    # full       = temporal serialisation + D1 + D2 + parity labels  (proposed model)
    # no_fair    = temporal serialisation + D1 only, real label mix   (ablation, RQ3)
    # mallm_flat = flattened (static) serialisation + D1 only + optimizer LLM
    #              that rewrites the guidelines = MALLM-GAN baseline (RQ3)
    mode: str = "full"
    cohort: str = "active"            # "active" (still enrolled at cutoff) | "all"

    backend: str = "hf"               # hf | vllm | openai | mock
    hf_model: str = "Qwen/Qwen2.5-7B-Instruct"
    load_in_4bit: bool = False
    allow_cpu: bool = False           # only for testing with a tiny model
    openai_model: str = "gpt-4o-mini"
    openai_base_url: str = "https://api.openai.com/v1"

    # ---- generation ----
    prompts_per_batch: int = 8        # prompts per GPU pass (16 ran out of memory on T4 x2)
    students_per_prompt: int = 8
    shots_per_cell: int = 2           # few-shot examples per S x Y cell (4 cells)
    temperature: float = 0.9
    top_p: float = 0.95

    # ---- adversarial loop ----
    max_rounds: int = 10
    min_rounds: int = 3
    pool_rounds: int = 3              # max rounds pooled for scoring
    min_eval_rows: int = 60
    tol_d1: float = 0.10
    tol_d2: float = 0.10
    patience: int = 2
    fairness_weight: float = 1.0      # forced to 0 outside mode="full"
    label_policy: str = "auto"        # auto | parity | real
    use_data_card: bool = True
    use_optimizer_llm: str = "auto"   # auto -> True only for mallm_flat

    # ---- final data ----
    n_synthetic: int = 2000
    zero_cap_margin: float = 0.01     # all-zero share may exceed real share by this (v3: 0.03)
    copy_quantile: float = 0.01       # novelty threshold = this quantile of real-real NN distance
    time_budget_h: float = 10.5       # whole run (Kaggle kills at 12 h)
    train_budget_frac: float = 0.35   # share of the budget the adversarial loop may use
    seed: int = 42

    # ---- v4 realism additions ----
    activity_plan: bool = True        # qz/sc week plan per requested student (temporal modes only)
    plan_min_recall: float = 0.5      # reject students that fill < this share of planned qz/sc weeks
    active_shot_frac: float = 0.5     # share of few-shot examples with quiz/assessment activity
    plan_mode: str = "constrained"    # constrained (decoder enforces plan) | template | text

    # derived (filled by resolve())
    fmt: str = ""
    optimizer: bool = False

    def resolve(self):
        assert self.mode in ("full", "no_fair", "mallm_flat"), self.mode
        self.fmt = "flat" if self.mode == "mallm_flat" else "temporal"
        if self.label_policy == "auto":
            self.label_policy = "parity" if self.mode == "full" else "real"
        if self.mode != "full":
            self.fairness_weight = 0.0
        self.optimizer = (self.mode == "mallm_flat") if self.use_optimizer_llm == "auto" \
            else bool(self.use_optimizer_llm)
        if self.fmt != "temporal":
            self.activity_plan = False
        return self


def _log(cfg, msg):
    print(msg, flush=True)
    try:
        with open(os.path.join(cfg.OUT_DIR, "run_log.txt"), "a") as fh:
            fh.write(time.strftime("%H:%M:%S ") + msg + "\n")
    except OSError:
        pass


# =============================================================================
# 1. SERIALISATION + STRICT PARSER
# =============================================================================
def serialize(x, m, s, y, fmt):
    W = int(m.sum())
    head = f"<S={int(s)};Y={int(y)};W={W}>"
    if fmt == "temporal":
        body = " | ".join(f"w{w}:" + ",".join(str(int(round(v))) for v in x[w])
                          for w in range(W))
    else:   # flat, feature-major "column=value" like GReaT / MALLM-GAN
        body = ", ".join(f"{TAGS[f]}_w{w}={int(round(x[w, f]))}"
                         for f in range(len(TAGS)) for w in range(W))
    return f"{head} {body} </end>"


_HEAD = re.compile(r"<\s*S\s*=\s*([01])\s*;\s*Y\s*=\s*([01])\s*;\s*W\s*=\s*(\d+)\s*>")
_WEEK = re.compile(r"w(\d+)\s*:\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)")
_FLAT = re.compile(r"(hp|sp|oc|qz|sc)_w(\d+)\s*=\s*(\d+)")


def parse_generation(text, fmt, allowed_W, clip_max):
    """Returns (rows, reasons). A row is (x, m, s, y, k), k = header index. Students are segmented by
    their header, so missing newlines do not break parsing."""
    rows, reasons = [], []
    heads = list(_HEAD.finditer(text))
    for k, h in enumerate(heads):
        seg = text[h.end(): heads[k + 1].start() if k + 1 < len(heads) else len(text)]
        seg = seg.split("</end>")[0]
        s, y, W = int(h.group(1)), int(h.group(2)), int(h.group(3))
        if W not in allowed_W:
            reasons.append("bad_W"); continue
        x = np.zeros((T_WEEKS, len(TAGS)), np.float32)
        m = np.zeros(T_WEEKS, np.int8)
        if fmt == "temporal":
            wk = _WEEK.findall(seg)
            idx = [int(a[0]) for a in wk]
            if idx[:W] != list(range(W)):          # extra weeks beyond W are ignored
                reasons.append("week_count"); continue
            for a in wk[:W]:
                x[int(a[0])] = [float(v) for v in a[1:]]
        else:
            cells = _FLAT.findall(seg)
            seen = set()
            for tag, w, v in cells:
                w = int(w)
                if w < W:
                    x[w, TAGS.index(tag)] = float(v); seen.add((tag, w))
            if len(seen) < len(TAGS) * W or any(int(w) >= W for _, w, _ in cells):
                reasons.append("week_count"); continue
        m[:W] = 1
        x = np.minimum(x, clip_max[None, :])
        x[:, SCORE_IDX] = np.minimum(x[:, SCORE_IDX], 100.0)
        rows.append((x, m, s, y, k))
    if not heads:
        reasons.append("no_header")
    return rows, reasons


# =============================================================================
# 2. LLM BACKENDS  (batched)
# =============================================================================
_WSTART = re.compile(r"w(\d+)\s*:")


def plan_token_ids(tok):
    """Token ids of the single digits and of all tokens whose text starts with ','."""
    d = {}
    for c in "0123456789":
        ids = tok.encode(c, add_special_tokens=False)
        assert len(ids) == 1, f"digit {c} is not a single token: {ids}"
        d[c] = ids[0]
    comma = []
    for t, i in tok.get_vocab().items():
        s = tok.convert_tokens_to_string([t])
        if s.startswith(",") and not any(ch.isdigit() for ch in s):
            comma.append(i)
    return dict(d0=[d["0"]], d19=[d[c] for c in "123456789"], digits=[d[c] for c in "0123456789"],
                comma=sorted(comma))


class PlanConstraint:
    """Logits processor for plan-constrained decoding (v4.2). Tracks the text each
    sequence has generated; inside week w of the k-th student (k = number of
    headers written so far) it forces the qz field (4th value) and the sc field
    (5th value) to follow plan k: unplanned -> exactly 0, planned -> first digit
    1-9 (sc at most 3 digits). Everything else is unconstrained."""
    def __init__(self, decode, plans, prompt_len, ids):
        self.decode, self.plans, self.L0 = decode, plans, prompt_len
        self.d0, self.d19, self.digits, self.comma = ids["d0"], ids["d19"], ids["digits"], ids["comma"]
        self.buf = [""] * len(plans); self.n = 0
        self.stats = dict(forced=0)

    def action(self, b):
        t = self.buf[b]
        k = t.count("<S=") - 1
        if k < 0 or k >= len(self.plans[b]) or self.plans[b][k] is None:
            return None
        seg = t[t.rfind("<S="):]
        m = None
        for m in _WSTART.finditer(seg):
            pass
        if m is None:
            return None
        after = seg[m.end():]
        if "|" in after or "<" in after or "\n" in after:
            return None
        parts = after.split(","); f = len(parts) - 1; cur = parts[-1].strip()
        if not cur.isdigit() and cur != "":
            return None
        w = int(m.group(1)); pl = self.plans[b][k]; q, a = pl[0], pl[1]
        qn = pl[2] if len(pl) > 2 else {}; an = pl[3] if len(pl) > 3 else {}
        if f == 3:
            if w in q:
                n = qn.get(w)
                if cur == "":
                    return ("only", self.d19)
                if n is None:
                    return None
                return ("only", self.digits) if len(cur) < n else ("only", self.comma)
            return ("only", self.d0) if cur == "" else ("only", self.comma)
        if f == 4:
            if w in a:
                n = an.get(w, 3)
                if cur == "":
                    return ("only", self.d19)
                if len(cur) < n and w in an:
                    return ("only", self.digits)
                return ("ban", self.digits) if len(cur) >= n else None
            return ("only", self.d0) if cur == "" else ("ban", self.digits + self.comma)
        return None

    def __call__(self, input_ids, scores):
        import torch
        gen = input_ids[:, self.L0:]
        if gen.shape[1] > self.n:
            new = gen[:, self.n:].tolist(); self.n = gen.shape[1]
            for b, ids in enumerate(new):
                self.buf[b] += self.decode(ids)
        for b in range(len(self.buf)):
            act = self.action(b)
            if act is None:
                continue
            kind, ids = act
            self.stats["forced"] += 1
            if kind == "only":
                keep = scores[b, ids].clone()
                scores[b, :] = -float("inf"); scores[b, ids] = keep
            else:
                scores[b, ids] = -float("inf")
        return scores


class HFGenerator:
    def __init__(self, cfg):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.torch, self.cfg = torch, cfg
        cpu = not torch.cuda.is_available()
        if cpu and not cfg.allow_cpu:
            raise RuntimeError("No GPU. On Kaggle: Settings > Accelerator > GPU T4 x2.")
        _log(cfg, f"[LLM] HF {cfg.hf_model} on " + ("CPU (test only)" if cpu else
                  f"{torch.cuda.get_device_name(0)} x{torch.cuda.device_count()}"))
        self.tok = AutoTokenizer.from_pretrained(cfg.hf_model, padding_side="left")
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        kw = dict(device_map=None if cpu else "auto")
        dtype = torch.float32 if cpu else torch.float16
        if cfg.load_in_4bit and not cpu:
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        import transformers
        from packaging.version import Version
        new_api = Version(transformers.__version__) >= Version("4.56")
        kw["dtype" if new_api else "torch_dtype"] = dtype     # avoid silent float32 load
        self.model = AutoModelForCausalLM.from_pretrained(cfg.hf_model, **kw)
        self.model.eval()
        devs = sorted(set(str(v) for v in getattr(self.model, "hf_device_map", {}).values()))
        _log(cfg, f"[LLM] transformers {transformers.__version__}, dtype={self.model.dtype}, "
                  f"devices={devs or [str(self.model.device)]}")
        if any("cpu" in d_ or "disk" in d_ for d_ in devs):
            _log(cfg, "[LLM] WARNING: part of the model is offloaded to CPU/disk -> very slow. "
                      "Check that the accelerator is 'GPU T4 x2'.")
        self.bs = cfg.prompts_per_batch
        self.tokens_out = 0; self.seconds = 0.0
        self.plan_ids = plan_token_ids(self.tok)
        _log(cfg, f"[LLM] plan constraint tokens: digits={self.plan_ids['digits']} "
                  f"comma-start tokens={len(self.plan_ids['comma'])}")

    def n_tokens(self, text):
        return len(self.tok(text)["input_ids"])

    def _prompt(self, system, user):
        return self.tok.apply_chat_template(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=True)

    def chat_batch(self, system, users, max_new_tokens, plans=None):
        torch = self.torch
        from transformers import LogitsProcessorList
        prompts = [self._prompt(system, u) for u in users]
        outs, i = [], 0
        while i < len(prompts):
            chunk = prompts[i:i + self.bs]
            try:
                t0 = time.time()
                enc = self.tok(chunk, return_tensors="pt", padding=True).to(self.model.device)
                lp = None
                if plans is not None:
                    dec = lambda ids: self.tok.decode(ids, skip_special_tokens=True)
                    lp = LogitsProcessorList([PlanConstraint(dec, plans[i:i + len(chunk)],
                                                             enc["input_ids"].shape[1], self.plan_ids)])
                with torch.inference_mode():
                    gen = self.model.generate(
                        **enc, max_new_tokens=max_new_tokens, do_sample=True,
                        temperature=self.cfg.temperature, top_p=self.cfg.top_p,
                        pad_token_id=self.tok.pad_token_id, logits_processor=lp)
                new = gen[:, enc["input_ids"].shape[1]:]
                self.tokens_out += int((new != self.tok.pad_token_id).sum())
                self.seconds += time.time() - t0
                outs += self.tok.batch_decode(new, skip_special_tokens=True)
                i += len(chunk)
            except torch.OutOfMemoryError if hasattr(torch, "OutOfMemoryError") else torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if self.bs == 1:
                    raise
                self.bs = max(1, self.bs // 2)
                _log(self.cfg, f"[LLM] CUDA OOM -> prompts_per_batch reduced to {self.bs}")
        return outs


class VLLMGenerator:
    def __init__(self, cfg):
        import torch
        from vllm import LLM, SamplingParams
        from transformers import AutoTokenizer
        self.cfg, self.SP = cfg, SamplingParams
        self.tok = AutoTokenizer.from_pretrained(cfg.hf_model)
        self.llm = LLM(model=cfg.hf_model, dtype="half",
                       tensor_parallel_size=max(1, torch.cuda.device_count()),
                       gpu_memory_utilization=0.90, max_model_len=8192, seed=cfg.seed)
        self.tokens_out = 0; self.seconds = 0.0
        _log(cfg, "[LLM] vLLM backend ready")

    def n_tokens(self, text):
        return len(self.tok(text)["input_ids"])

    def chat_batch(self, system, users, max_new_tokens, plans=None):
        prompts = [self.tok.apply_chat_template(
            [{"role": "system", "content": system}, {"role": "user", "content": u}],
            tokenize=False, add_generation_prompt=True) for u in users]
        t0 = time.time()
        res = self.llm.generate(prompts, self.SP(
            temperature=self.cfg.temperature, top_p=self.cfg.top_p,
            max_tokens=max_new_tokens))
        self.seconds += time.time() - t0
        outs = [r.outputs[0].text for r in res]
        self.tokens_out += sum(len(r.outputs[0].token_ids) for r in res)
        return outs


class OpenAIGenerator:
    def __init__(self, cfg):
        from openai import OpenAI
        self.cfg = cfg
        self.client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""),
                             base_url=cfg.openai_base_url)
        self.tokens_out = 0; self.seconds = 0.0

    def n_tokens(self, text):
        return len(text) // 3

    def chat_batch(self, system, users, max_new_tokens, plans=None):
        outs = []
        for u in users:
            t0 = time.time()
            r = self.client.chat.completions.create(
                model=self.cfg.openai_model, temperature=self.cfg.temperature,
                max_tokens=max_new_tokens,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": u}])
            self.seconds += time.time() - t0
            outs.append(r.choices[0].message.content)
        return outs


class MockGenerator:
    """CPU test double: answers each requested header with a perturbed real
    student of the same S, Y, W (and some deliberately broken lines), so the
    whole loop can be tested without a GPU. NOT for thesis results."""
    def __init__(self, cfg, ctx):
        self.cfg, self.ctx = cfg, ctx
        self.rng = np.random.default_rng(cfg.seed + 7)
        self.tokens_out = 0; self.seconds = 0.0

    def n_tokens(self, text):
        return len(text) // 3

    def chat_batch(self, system, users, max_new_tokens, plans=None):
        if "prompt optimizer" in system:
            return ["- Match the weekly click levels of the real rows.\n- Keep zeros realistic."] * len(users)
        outs = []
        d, tr = self.ctx["data"], self.ctx["tr"]
        for ui, u in enumerate(users):
            lines = []
            tail = u.split("in this order")[-1]
            given = plans[ui] if plans is not None else None
            plans_u = []
            for ln in tail.splitlines():
                if ln.startswith("<S="):
                    wk = re.findall(r"w(\d+):_,_,_,(Q|0),(S|0)", ln)
                    plans_u.append((" ".join(f"w{a}" for a, q, s_ in wk if q == "Q"),
                                    " ".join(f"w{a}" for a, q, s_ in wk if s_ == "S")))
            if given is not None:
                plans_u = [(" ".join(f"w{w}" for w in pq), " ".join(f"w{w}" for w in pa))
                           for pq, pa, *_ in given]
            for hk, h in enumerate(_HEAD.finditer(tail)):
                s, y, W = int(h.group(1)), int(h.group(2)), int(h.group(3))
                pool = tr[(d["S"][tr] == s) & (d["y"][tr] == y) & (d["mask"][tr].sum(1) == W)]
                if len(pool) == 0:
                    continue
                i = self.rng.choice(pool)
                x = d["X"][i] * self.rng.lognormal(0, 0.35, d["X"][i].shape)
                if hk < len(plans_u):                 # follow the plan (mostly)
                    wk = lambda t: [int(v) for v in re.findall(r"w(\d+)", t)]
                    x[:, 3] = 0; x[:, SCORE_IDX] = 0
                    for w in wk(plans_u[hk][0]):
                        if self.rng.random() < 0.9: x[w, 3] = self.rng.integers(1, 40)
                    for w in wk(plans_u[hk][1]):
                        if self.rng.random() < 0.9: x[w, SCORE_IDX] = self.rng.integers(40, 100)
                if self.rng.random() < 0.05:          # broken line -> must be rejected
                    lines.append(f"<S={s};Y={y};W={W}> w0:1,2,3 | w1:4 </end>")
                    continue
                lines.append(serialize(np.round(x), d["mask"][i], s, y, self.cfg.fmt))
            outs.append("\n".join(lines))
        return outs


def make_generator(cfg, ctx):
    if cfg.backend == "mock":
        return MockGenerator(cfg, ctx)
    if cfg.backend == "openai":
        return OpenAIGenerator(cfg)
    if cfg.backend == "vllm":
        try:
            return VLLMGenerator(cfg)
        except Exception as e:                 # fall back, never crash the run
            _log(cfg, f"[LLM] vLLM unavailable ({type(e).__name__}: {e}) -> using HF")
    return HFGenerator(cfg)


# =============================================================================
# 3. REAL-DATA CONTEXT (reference sample, data card, novelty filter)
# =============================================================================
def build_context(cfg):
    d = load_real(cfg.DATA_DIR)
    tr, va, te = get_split(d, cfg.cohort, cfg.seed)
    X, M, y, S = d["X"], d["mask"], d["y"], d["S"]
    ctx = dict(data=d, tr=tr, va=va)
    ctx["real_vecs"] = build_matrix(X[tr], M[tr])
    ctx["cell_idx"] = {(s, yy): np.where((S[tr] == s) & (y[tr] == yy))[0]
                       for s in (0, 1) for yy in (0, 1)}
    W = M[tr].sum(1).astype(int)
    ctx["allowed_W"] = sorted(set(W.tolist()))
    ctx["W_marg"] = {w: float((W == w).mean()) for w in ctx["allowed_W"]}
    ctx["W_cell"] = {c: {w: float((W[ix] == w).mean()) for w in ctx["allowed_W"]}
                     for c, ix in ctx["cell_idx"].items()}
    ctx["clip_max"] = X[tr].reshape(-1, X.shape[2]).max(0)
    f_rate = float(S[tr].mean()); p = float(y[tr].mean())
    if cfg.label_policy == "parity":
        py = {0: p, 1: p}
    else:
        py = {s: float(y[tr][S[tr] == s].mean()) for s in (0, 1)}
    ctx["cell_prob"] = {(s, yy): (f_rate if s else 1 - f_rate) * (py[s] if yy else 1 - py[s])
                        for s in (0, 1) for yy in (0, 1)}
    tot = X[tr].sum((1, 2))
    ctx["zero_share"] = {yy: float((tot[y[tr] == yy] == 0).mean()) for yy in (0, 1)}
    # real D2 reference (how predictable gender is in REAL data, same feature set)
    rng = np.random.default_rng(cfg.seed)
    sub = rng.choice(len(tr), min(2000, len(tr)), replace=False)
    ctx["real_D2_auc"] = score_D2(ctx["real_vecs"][sub], S[tr][sub], y[tr][sub], cfg.seed)[0]
    # novelty filter threshold: low quantile of real(val)->real(train) NN distance
    ctx["real_flat"] = flat_log(X[tr], M[tr])
    ctx["nn"] = NearestNeighbors(n_neighbors=1).fit(ctx["real_flat"])
    vf = flat_log(X[va], M[va])
    rich = (vf > 0).sum(1) >= 5
    dist = ctx["nn"].kneighbors(vf[rich])[0][:, 0]
    ctx["copy_thr"] = float(np.quantile(dist, cfg.copy_quantile))
    ctx["data_card"] = data_card(X[tr], M[tr], y[tr])
    # v4: donors for activity plans, active few-shot pools, real activity rates
    Xt, Mt = X[tr], M[tr].astype(bool)
    has_act = ((Xt[:, :, 3] > 0) | (Xt[:, :, SCORE_IDX] > 0)).any(1)
    ctx["y_idx"] = {yy: np.where(y[tr] == yy)[0] for yy in (0, 1)}
    ctx["active_cell_idx"] = {c: ix[has_act[ix]] for c, ix in ctx["cell_idx"].items()}
    ctx["real_rates"] = dict(qz=float(100 * (Xt[:, :, 3][Mt] > 0).mean()),
                             sc=float(100 * (Xt[:, :, SCORE_IDX][Mt] > 0).mean()))
    _log(cfg, f"[data] cohort={cfg.cohort} train={len(tr)} pos={p:.3f} female={f_rate:.3f} "
              f"label_policy={cfg.label_policy} real_D2_auc={ctx['real_D2_auc']:.3f} "
              f"copy_thr={ctx['copy_thr']:.3f}")
    return ctx


def data_card(X, M, y):
    """Aggregate (non-identifying) description of the real training data."""
    names = ["homepage clicks (hp)", "subpage clicks (sp)", "course-content clicks (oc)",
             "quiz interactions (qz)"]
    out = ["REAL DATA SUMMARY (training students; statistics per real week):"]
    for yy, lab in [(0, "Y=0 (passed)"), (1, "Y=1 (withdrew or failed)")]:
        Xi, Mi = X[y == yy], M[y == yy].astype(bool)
        out.append(f"{lab}:")
        for f, nm in enumerate(names):
            v = Xi[:, :, f][Mi]; nz = v[v > 0]
            out.append(f"  - {nm}: mean {v.mean():.1f}/week, non-zero in {100*(v>0).mean():.0f}% "
                       f"of weeks, typical non-zero value {np.median(nz):.0f} "
                       f"(90th percentile {np.percentile(nz, 90):.0f})")
        v = Xi[:, :, SCORE_IDX][Mi]; nz = v[v > 0]
        out.append(f"  - assessment score (sc): submitted in {100*(v>0).mean():.0f}% of weeks, "
                   f"mean score when submitted {nz.mean():.0f}")
        wk = [Xi[:, w, 0][Mi[:, w]].mean() for w in range(X.shape[1])]
        out.append("  - mean hp by week: " + ", ".join(f"w{w}={v:.0f}" for w, v in enumerate(wk)))
        qw = [100 * (Xi[:, w, 3][Mi[:, w]] > 0).mean() for w in range(X.shape[1])]
        out.append("  - % with quiz activity by week: " +
                   ", ".join(f"w{w}={v:.0f}" for w, v in enumerate(qw)))
        sw = [100 * (Xi[:, w, SCORE_IDX][Mi[:, w]] > 0).mean() for w in range(X.shape[1])]
        out.append("  - % with an assessment by week: " +
                   ", ".join(f"w{w}={v:.0f}" for w, v in enumerate(sw)))
        tot = Xi.sum((1, 2))
        out.append(f"  - students with no activity at all: {100*(tot==0).mean():.0f}%")
    out.append("Counts are skewed: many small values, a few large ones. "
               "Activity of a student is correlated across hp, sp, oc in the same week.")
    return "\n".join(out)


class NoveltyFilter:
    """Rejects (a) near-copies of real training students and (b) duplicates of
    already accepted synthetic students - gender is ignored, so gender-swapped
    twins are caught. Students with < min_nonzero non-zero values (e.g. all
    zero) are not identifying and are exempt."""
    def __init__(self, ctx, min_nonzero=5):
        self.nn, self.thr, self.mnz = ctx["nn"], ctx["copy_thr"], min_nonzero
        self.syn = np.zeros((0, ctx["real_flat"].shape[1]), np.float32)

    def check(self, x, m):
        v = flat_log(x[None], m[None])[0]
        if (v > 0).sum() < self.mnz:
            return None, v
        if self.nn.kneighbors(v[None])[0][0, 0] < self.thr:
            return "copy_of_real", v
        if len(self.syn) and np.sqrt(((self.syn - v) ** 2).sum(1)).min() < self.thr:
            return "duplicate_synthetic", v
        return None, v

    def add(self, v):
        self.syn = np.vstack([self.syn, v[None]])


# =============================================================================
# 4. DISCRIMINATORS  (held-out, cross-validated)
# =============================================================================
def _clf():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0))


def score_D1(ctx, sv, sS, sy, seed):
    """Real vs synthetic. Real sample has the SAME S x Y composition as the
    synthetic sample, so D1 judges realism, not the label mix.
    Returns CV accuracy and the 3 most discriminating features (+ direction)."""
    rng = np.random.default_rng(seed)
    r = []
    for (s, yy), ix in ctx["cell_idx"].items():
        k = int(((sS == s) & (sy == yy)).sum())
        if k:
            r.append(ctx["real_vecs"][rng.choice(ix, k, replace=len(ix) < k)])
    rv = np.vstack(r)
    X = np.vstack([rv, sv]); lab = np.r_[np.ones(len(rv)), np.zeros(len(sv))]
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    acc = float(accuracy_score(lab, cross_val_predict(_clf(), X, lab, cv=cv)))
    coef = _clf().fit(X, lab)[-1].coef_[0]
    top = [(int(i), "higher" if coef[i] > 0 else "lower") for i in np.argsort(-np.abs(coef))[:3]]
    return acc, top


def score_D2(sv, S, y, seed):
    """Predict gender from synthetic features + label; CV AUC (FairGAN idea)."""
    S = np.asarray(S).astype(int)
    if len(np.unique(S)) < 2 or np.bincount(S).min() < 5:
        return 1.0, 0
    X = np.column_stack([sv, y])
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    p = cross_val_predict(_clf(), X, S, cv=cv, method="predict_proba")[:, 1]
    coef = _clf().fit(X, S)[-1].coef_[0][:sv.shape[1]]
    return float(roc_auc_score(S, p)), int(np.argmax(np.abs(coef)))


# =============================================================================
# 5. FEEDBACK (specific, numeric)
# =============================================================================
def _stat_of_vec(i):
    if i >= 5 * len(STATS):
        return None, None
    return i // len(STATS), STATS[i % len(STATS)]


def _fmt(v):
    return f"{v:.2f}" if abs(v) < 1 else f"{v:.1f}"


def realism_feedback(ctx, d1, top, sx, sm, sy, tol):
    d = ctx["data"]; tr = ctx["tr"]
    X, M, y = d["X"][tr], d["mask"][tr], d["y"][tr]
    if abs(d1 - 0.5) <= tol:
        head = f"- REALISM: good (D1 accuracy {d1:.2f}, target 0.50). Keep this style."
    else:
        head = (f"- REALISM: a classifier still separates real from your students "
                f"(D1 accuracy {d1:.2f}, target 0.50). Fix these differences:")
    lines = [head]
    for i, direction in top:                       # what D1 relies on most
        f, st = _stat_of_vec(i)
        if f is None:
            continue
        r = raw_stat(X, M, f, st).mean(); s = raw_stat(sx, sm, f, st).mean()
        lines.append(f"  * {FEATURES[f]} ({st}): real {_fmt(r)} vs yours {_fmt(s)}")
    # largest relative gaps of simple per-label statistics
    gaps = []
    for yy in (0, 1):
        rm, smm = M[y == yy].astype(bool), sm[sy == yy].astype(bool)
        if smm.sum() == 0:
            continue
        for f in range(len(FEATURES)):
            rv, svv = X[y == yy][:, :, f][rm], sx[sy == yy][:, :, f][smm]
            for nm, a, b in [("mean per week", rv.mean(), svv.mean()),
                             ("% of weeks non-zero", 100 * (rv > 0).mean(), 100 * (svv > 0).mean())]:
                gaps.append((abs(math.log((b + 0.05 * a + 0.01) / (a + 0.05 * a + 0.01))),
                             f"  * Y={yy} {FEATURES[f]} {nm}: real {_fmt(a)} vs yours {_fmt(b)}"))
        rz = ctx["zero_share"][yy] * 100
        sz = 100 * (sx[sy == yy].sum((1, 2)) == 0).mean()
        gaps.append((abs(math.log((sz + 1) / (rz + 1))),
                     f"  * Y={yy} students with no activity at all: real {rz:.0f}% vs yours {sz:.0f}%"))
    gaps.sort(key=lambda g: -g[0])
    lines += [g[1] for g in gaps[:5] if g[0] > 0.15]
    return "\n".join(lines)


def fairness_feedback(ctx, d2, li, sx, sm, sS, sy, tol):
    if abs(d2 - 0.5) <= tol:
        return (f"- FAIRNESS: good (D2 AUC {d2:.2f}, target 0.50). Students with the same Y "
                f"must keep looking the same for S=0 and S=1.")
    f, st = _stat_of_vec(li)
    lines = [f"- FAIRNESS: gender can still be predicted from your students "
             f"(D2 AUC {d2:.2f}, target 0.50)."]
    if f is not None:
        for yy in (0, 1):
            a = raw_stat(sx[(sS == 0) & (sy == yy)], sm[(sS == 0) & (sy == yy)], f, st)
            b = raw_stat(sx[(sS == 1) & (sy == yy)], sm[(sS == 1) & (sy == yy)], f, st)
            if len(a) and len(b):
                lines.append(f"  * Y={yy}: {FEATURES[f]} ({st}) is {_fmt(a.mean())} for S=0 "
                             f"but {_fmt(b.mean())} for S=1")
        lines.append(f"  Give S=0 and S=1 students with the same Y the same distribution "
                     f"of {FEATURES[f]}; S must not change any number.")
    return "\n".join(lines)


def plan_of(x, m):
    """Weeks with quiz activity and with a submitted assessment, plus the number of
    digits of the donor's value in those weeks (v4.3: order of magnitude only)."""
    W = int(m.sum())
    q = [w for w in range(W) if x[w, 3] > 0]
    a = [w for w in range(W) if x[w, SCORE_IDX] > 0]
    nd = lambda v: len(str(max(int(round(v)), 1)))
    return (q, a, {w: nd(x[w, 3]) for w in q}, {w: min(nd(x[w, SCORE_IDX]), 3) for w in a})


def plan_text(plan):
    f = lambda L: ",".join(f"w{w}" for w in L) if L else "none"
    return f"(plan: qz in {f(plan[0])}; sc in {f(plan[1])})"


def plan_template(s, y, W, plan):
    q, a = set(plan[0]), set(plan[1])
    body = " | ".join(f"w{w}:_,_,_,{'Q' if w in q else 0},{'S' if w in a else 0}" for w in range(W))
    return f"<S={s};Y={y};W={W}> {body} </end>"


def plan_recall(x, plan):
    """Share of planned qz/sc weeks that the student actually filled (1.0 if none planned),
    and number of qz/sc weeks with activity outside the plan."""
    q, a = plan[0], plan[1]
    n = len(q) + len(a)
    hit = sum(x[w, 3] > 0 for w in q) + sum(x[w, SCORE_IDX] > 0 for w in a)
    extra = int(sum(x[w, 3] > 0 for w in range(len(x)) if w not in q) +
                sum(x[w, SCORE_IDX] > 0 for w in range(len(x)) if w not in a))
    return (hit / n if n else 1.0), extra


def activity_feedback(ctx, sx, sm, rej):
    Mb = sm.astype(bool)
    q = 100 * (sx[:, :, 3][Mb] > 0).mean(); a = 100 * (sx[:, :, SCORE_IDX][Mb] > 0).mean()
    rr = ctx["real_rates"]
    lines = [f"- ACTIVITY: quiz interactions are non-zero in {q:.0f}% of your weeks (real {rr['qz']:.0f}%), "
             f"assessment scores in {a:.0f}% (real {rr['sc']:.0f}%)."]
    if rej.get("plan_n"):
        rc = rej["plan_recall_sum"] / rej["plan_n"]
        lines.append(f"  * You filled {100 * rc:.0f}% of the planned qz/sc weeks; "
                     f"{rej.get('plan_mismatch', 0)} students ignored their plan and were discarded. "
                     f"Every planned qz week needs qz > 0 and every planned sc week a score 1-100.")
    return "\n".join(lines)


OPT_SYSTEM = ("You are a prompt optimizer in a GAN-style loop. A generator LLM receives "
              "GUIDELINES plus real example rows and writes synthetic student rows. A "
              "discriminator then tries to tell synthetic rows from real rows.")


def optimize_guidelines(gen, guidelines, history, feedback):
    hist = "\n".join(f"- round {h['round']}: D1 accuracy {h['D1_acc']:.2f}" for h in history[-3:])
    user = (f"Current guidelines:\n{guidelines or '(none yet)'}\n\nHistory (D1 accuracy, "
            f"0.50 is perfect):\n{hist or '(none)'}\n\nDiscriminator report on the latest "
            f"rows:\n{feedback}\n\nWrite improved GUIDELINES (bullet points, at most 150 words) "
            f"so the next rows are indistinguishable from real rows. Output only the guidelines.")
    return gen.chat_batch(OPT_SYSTEM, [user], 400)[0].strip()


# =============================================================================
# 6. PROMPTS
# =============================================================================
def system_prompt(cfg, ctx):
    if cfg.fmt == "temporal":
        fmt = ("FORMAT - one student per line:\n"
               "<S=s;Y=y;W=n> w0:hp,sp,oc,qz,sc | w1:hp,sp,oc,qz,sc | ... | "
               "w(n-1):hp,sp,oc,qz,sc </end>\n"
               "- Write exactly W weeks, numbered w0 .. w(W-1), five integers per week.\n")
        if cfg.activity_plan and cfg.plan_mode == "template":
            fmt += ("- You get one TEMPLATE line per student. Copy the template and replace every "
                    "placeholder by a number, keep every 0 that is already written:\n"
                    "    _  -> clicks (integer >= 0; active students click in most weeks)\n"
                    "    Q  -> quiz interactions, an integer >= 1\n"
                    "    S  -> assessment score, an integer from 1 to 100\n"
                    "  Example: template  <S=0;Y=0;W=3> w0:_,_,_,0,0 | w1:_,_,_,Q,0 | w2:_,_,_,Q,S </end>\n"
                    "           answer    <S=0;Y=0;W=3> w0:14,3,22,0,0 | w1:9,0,15,31,0 | w2:25,11,40,12,84 </end>\n"
                    "  Clicks (hp, sp, oc) are usually higher in weeks with Q or S.\n")
    else:
        fmt = ("FORMAT - one student per line, as column=value pairs:\n"
               "<S=s;Y=y;W=n> hp_w0=.., hp_w1=.., ..., hp_w(n-1)=.., sp_w0=.., ..., "
               "sc_w(n-1)=.. </end>\n"
               "- Give every column hp,sp,oc,qz,sc for every week index 0 .. W-1.\n")
    rules = ("You generate synthetic records of Open University (OULAD) students for the first "
             "weeks of a course (early-warning window).\n" + fmt +
             "- S = gender code (0/1). Y = final outcome: 1 = withdrew or failed, 0 = passed.\n"
             "- hp = homepage clicks, sp = subpage clicks, oc = course-content clicks, "
             "qz = quiz interactions in that week (integers >= 0).\n"
             "- sc = mean score (0-100) of assessments submitted in that week, 0 if none.\n"
             "- Output ONLY the student lines: no numbering, no explanations, no code fences.\n"
             "- Every student must be NEW and different; never copy or lightly edit an example.\n")
    if cfg.mode == "full":
        rules += ("- FAIRNESS: S must not influence the numbers. Students with the same Y must "
                  "look statistically identical for S=0 and S=1.\n")
    if cfg.use_data_card:
        rules += "\n" + ctx["data_card"] + "\n"
    return rules


def sample_shots(ctx, cfg, rng):
    d, tr = ctx["data"], ctx["tr"]
    idx = []
    for c, ix in ctx["cell_idx"].items():
        n_act = int(round(cfg.shots_per_cell * cfg.active_shot_frac)) if cfg.activity_plan else 0
        act = ctx["active_cell_idx"][c]
        pick = list(rng.choice(act, min(n_act, len(act)), replace=False)) if n_act else []
        rest = [i for i in rng.permutation(ix)[:cfg.shots_per_cell * 3] if i not in pick]
        pick += rest[:cfg.shots_per_cell - len(pick)]
        idx += list(tr[np.array(pick, int)])
    rng.shuffle(idx)
    out = []
    for i in idx:
        line = serialize(d["X"][i], d["mask"][i], d["S"][i], d["y"][i], cfg.fmt)
        out.append(line)
    return out


def user_prompt(shots, slots, guidance, cfg):
    g = ""
    if guidance:
        title = "GUIDELINES" if cfg.optimizer else "FEEDBACK FROM THE DISCRIMINATORS (apply it)"
        g = f"\n{title}:\n{guidance}\n"
    if slots and slots[0][3] is not None and cfg.plan_mode == "template":
        tpl = "\n".join(plan_template(s, y, w, pl) for s, y, w, pl in slots)
        return ("REAL EXAMPLE STUDENTS:\n" + "\n".join(shots) + "\n" + g +
                f"\nWrite {len(slots)} NEW students, one per line, by filling these templates "
                f"in this order (replace every _, Q and S by a number):\n{tpl}")
    heads = "\n".join(f"<S={s};Y={y};W={w}>" for s, y, w, pl in slots)
    return ("REAL EXAMPLE STUDENTS:\n" + "\n".join(shots) + "\n" + g +
            f"\nWrite {len(slots)} NEW students, one per line, using exactly these headers "
            f"in this order:\n{heads}")


def draw_slots(ctx, cfg, rng, n, need=None):
    """need=None: draw cells from the target distribution; otherwise from the
    remaining per-cell need (final generation balances itself)."""
    cells = list(ctx["cell_prob"].keys())
    if need is None:
        p = np.array([ctx["cell_prob"][c] for c in cells])
    else:
        p = np.array([max(need.get(c, 0), 0) for c in cells], float)
        if p.sum() == 0:
            p = np.array([ctx["cell_prob"][c] for c in cells])
    p = p / p.sum()
    out = []
    d, tr = ctx["data"], ctx["tr"]
    for k in rng.choice(len(cells), n, p=p):
        s, yy = cells[k]
        if cfg.activity_plan:
            # v4: plan (and length W) of a real donor with the same Y; in parity mode the
            # donor's gender is ignored so the plan carries no gender information
            pool = ctx["y_idx"][yy] if cfg.label_policy == "parity" else ctx["cell_idx"][(s, yy)]
            i = tr[rng.choice(pool)]
            out.append((s, yy, int(d["mask"][i].sum()), plan_of(d["X"][i], d["mask"][i])))
            continue
        wd = ctx["W_marg"] if cfg.label_policy == "parity" else ctx["W_cell"][(s, yy)]
        ws, wp = list(wd.keys()), np.array(list(wd.values()))
        out.append((s, yy, int(rng.choice(ws, p=wp / wp.sum())), None))
    return out


# =============================================================================
# 7. ONE GENERATION STEP (batched prompts -> parsed -> filtered rows)
# =============================================================================
def generate_step(ctx, cfg, gen, guidance, rng, novelty, rej, need=None, accept=None):
    users, slot_lists = [], []
    for _ in range(cfg.prompts_per_batch):
        slots = draw_slots(ctx, cfg, rng, cfg.students_per_prompt, need)
        slot_lists.append(slots)
        users.append(user_prompt(sample_shots(ctx, cfg, rng), slots, guidance, cfg))
    plans = None
    if cfg.activity_plan and cfg.plan_mode == "constrained":
        plans = [[sl[3] for sl in slots] for slots in slot_lists]
    raw = gen.chat_batch(ctx["system"], users, ctx["max_new_tokens"], plans=plans)
    with open(os.path.join(cfg.OUT_DIR, "raw_samples.txt"), "a") as fh:   # for debugging
        fh.write(f"\n===== {time.strftime('%H:%M:%S')} (first output of this batch) =====\n"
                 + raw[0][:3000] + "\n")
    rows = []
    for text, slots in zip(raw, slot_lists):
        parsed, reasons = parse_generation(text, cfg.fmt, ctx["allowed_W"], ctx["clip_max"])
        for r in reasons:
            rej[r] = rej.get(r, 0) + 1
        for x, m, s, y, k in parsed:
            if cfg.activity_plan:
                sl = slots[k] if k < len(slots) else None
                if sl is None or sl[:3] != (s, y, int(m.sum())):
                    rej["plan_unmatched_header"] = rej.get("plan_unmatched_header", 0) + 1
                else:
                    rc, extra = plan_recall(x, sl[3])
                    rej["plan_n"] = rej.get("plan_n", 0) + 1
                    rej["plan_recall_sum"] = rej.get("plan_recall_sum", 0.0) + float(rc)
                    rej["plan_extra_weeks"] = rej.get("plan_extra_weeks", 0) + extra
                    if rc < cfg.plan_min_recall:
                        rej["plan_mismatch"] = rej.get("plan_mismatch", 0) + 1
                        continue
            why, v = novelty.check(x, m)
            if why is None and accept is not None:
                why = accept(x, m, s, y)
            if why:
                rej[why] = rej.get(why, 0) + 1
                continue
            novelty.add(v)
            rows.append((x, m, s, y))
    rej["requested"] = rej.get("requested", 0) + cfg.prompts_per_batch * cfg.students_per_prompt
    rej["accepted"] = rej.get("accepted", 0) + len(rows)
    return rows


def _row_json(x, m, s, y, **extra):
    return json.dumps(dict(x=np.asarray(x).tolist(), m=np.asarray(m).tolist(),
                           s=int(s), y=int(y), **extra))


def _row_load(l):
    r = json.loads(l)
    return (np.array(r["x"], np.float32), np.array(r["m"], np.int8), r["s"], r["y"]), r


def _stack(rows):
    return (np.stack([r[0] for r in rows]), np.stack([r[1] for r in rows]),
            np.array([r[2] for r in rows]), np.array([r[3] for r in rows]))


# =============================================================================
# 8. ADVERSARIAL PROMPT OPTIMISATION
# =============================================================================
def _p(cfg, name): return os.path.join(cfg.OUT_DIR, name)


def train_loop(ctx, cfg, gen, t_start):
    rng = np.random.default_rng(cfg.seed)
    st = json.load(open(_p(cfg, "train_state.json"))) if os.path.exists(_p(cfg, "train_state.json")) else None
    if st and st.get("finished"):
        _log(cfg, "[train] finished state found -> skipping the adversarial loop")
        return st
    rounds = {}
    if os.path.exists(_p(cfg, "round_rows.jsonl")):
        for l in open(_p(cfg, "round_rows.jsonl")):
            row, meta = _row_load(l); rounds.setdefault(meta["round"], []).append(row)
    if not st:
        st = dict(next_round=0, guidance=None, history=[], best_score=1e9, best_guidance=None,
                  best_round=None, streak=0, converged=False, finished=False, rejections={})
    else:
        _log(cfg, f"[train] RESUMING at round {st['next_round']}")
    novelty = NoveltyFilter(ctx)
    budget = cfg.time_budget_h * 3600 * cfg.train_budget_frac

    for r in range(st["next_round"], cfg.max_rounds):
        if time.time() - t_start > budget:
            _log(cfg, f"[train] training time budget used up at round {r}"); break
        t0 = time.time(); rej = {}
        rows = generate_step(ctx, cfg, gen, st["guidance"], rng, novelty, rej)
        rounds[r] = rows
        with open(_p(cfg, "round_rows.jsonl"), "a") as fh:
            for x, m, s, y in rows:
                fh.write(_row_json(x, m, s, y, round=r) + "\n")
        pool, k = [], r
        while k >= 0 and r - k < cfg.pool_rounds and (k == r or len(pool) < cfg.min_eval_rows):
            pool += rounds.get(k, []); k -= 1
        for kk, vv in rej.items():
            st["rejections"][kk] = st["rejections"].get(kk, 0) + vv
        if len(pool) < cfg.min_eval_rows:
            _log(cfg, f"[round {r}] accepted={len(rows)} pooled={len(pool)} < {cfg.min_eval_rows}"
                      f" -> collecting more  rej={rej}")
            st["next_round"] = r + 1; json.dump(st, open(_p(cfg, "train_state.json"), "w"), indent=2)
            continue
        sx, sm, sS, sy = _stack(pool)
        sv = build_matrix(sx, sm)
        d1, top = score_D1(ctx, sv, sS, sy, cfg.seed + r)
        d2, li = score_D2(sv, sS, sy, cfg.seed + r)
        use_fair = cfg.mode == "full"
        score = abs(d1 - 0.5) + cfg.fairness_weight * abs(d2 - 0.5)
        ok = abs(d1 - 0.5) <= cfg.tol_d1 and (not use_fair or abs(d2 - 0.5) <= cfg.tol_d2)
        st["streak"] = st["streak"] + 1 if ok else 0
        Mb = sm.astype(bool)
        qz_pct = float(100 * (sx[:, :, 3][Mb] > 0).mean()); sc_pct = float(100 * (sx[:, :, SCORE_IDX][Mb] > 0).mean())
        plan_rc = rej["plan_recall_sum"] / rej["plan_n"] if rej.get("plan_n") else None
        h = dict(round=r, accepted=len(rows), pooled=len(pool), D1_acc=d1, D2_auc=d2,
                 qz_active_pct=qz_pct, sc_active_pct=sc_pct, plan_recall=plan_rc,
                 label_SPD=spd(sy, sS), leak=vec_name(li), D1_top=[vec_name(i) for i, _ in top],
                 score=score, in_band=ok, rejections=rej, minutes=(time.time() - t0) / 60,
                 tok_per_s=gen.tokens_out / max(gen.seconds, 1e-9))
        st["history"].append(h)
        _log(cfg, f"[round {r}] acc={len(rows)}/{rej['requested']} pooled={len(pool)} "
                  f"D1={d1:.3f} D2={d2:.3f} SPD={h['label_SPD']:+.3f} leak={h['leak']} "
                  f"qz%={qz_pct:.1f} sc%={sc_pct:.1f} plan={plan_rc if plan_rc is None else round(plan_rc, 2)} "
                  f"score={score:.3f} {'IN-BAND' if ok else ''} ({h['minutes']:.1f} min, "
                  f"{h['tok_per_s']:.0f} tok/s)")
        if score < st["best_score"]:     # guidance that PRODUCED this round's data
            st.update(best_score=score, best_guidance=st["guidance"], best_round=r)
        fb = realism_feedback(ctx, d1, top, sx, sm, sy, cfg.tol_d1)
        if cfg.activity_plan:
            fb += "\n" + activity_feedback(ctx, sx, sm, rej)
        if use_fair:
            fb += "\n" + fairness_feedback(ctx, d2, li, sx, sm, sS, sy, cfg.tol_d2)
        n_copy = rej.get("copy_of_real", 0) + rej.get("duplicate_synthetic", 0)
        if n_copy:
            fb += f"\n- DIVERSITY: {n_copy} of your students were copies/duplicates and were discarded. Write new, varied students."
        st["guidance"] = optimize_guidelines(gen, st["guidance"], st["history"], fb) if cfg.optimizer else fb
        st["next_round"] = r + 1
        json.dump(st, open(_p(cfg, "train_state.json"), "w"), indent=2)
        if st["streak"] >= cfg.patience and r + 1 >= cfg.min_rounds:
            st["converged"] = True
            _log(cfg, f"[train] CONVERGED at round {r} ({cfg.patience} consecutive rounds in band)")
            break
    if not st["converged"]:
        _log(cfg, f"[train] NOT converged. Using best guidance from round {st['best_round']} "
                  f"(score {st['best_score']:.3f}). Report this honestly.")
    st["finished"] = True
    json.dump(st, open(_p(cfg, "train_state.json"), "w"), indent=2)
    return st


# =============================================================================
# 9. FINAL GENERATION (balanced, filtered, resumable, time-boxed)
# =============================================================================
def final_generation(ctx, cfg, gen, guidance, t_start):
    rng = np.random.default_rng(cfg.seed + 999)
    path = _p(cfg, "synthetic_rows.jsonl")
    target = {c: int(round(cfg.n_synthetic * p)) for c, p in ctx["cell_prob"].items()}
    have = {c: 0 for c in target}
    zero_have = {0: 0, 1: 0}
    y_target = {yy: sum(v for (s, y2), v in target.items() if y2 == yy) for yy in (0, 1)}
    zero_cap = {yy: int(math.ceil((ctx["zero_share"][yy] + cfg.zero_cap_margin) * y_target[yy]))
                for yy in (0, 1)}
    novelty = NoveltyFilter(ctx)
    if os.path.exists(path):
        for l in open(path):
            (x, m, s, y), _ = _row_load(l)
            have[(s, y)] += 1; zero_have[y] += int(x.sum() == 0)
            novelty.add(flat_log(x[None], m[None])[0])
    rej = json.load(open(_p(cfg, "rejections_final.json"))) if os.path.exists(_p(cfg, "rejections_final.json")) else {}
    _log(cfg, f"[final] target per cell {target}, already {have}, zero cap {zero_cap}")

    def accept(x, m, s, y):          # last check; counts the row when accepted
        if have[(s, y)] >= target[(s, y)]:
            return "cell_full"
        if x.sum() == 0 and zero_have[y] >= zero_cap[y]:
            return "zero_cap"
        have[(s, y)] += 1; zero_have[y] += int(x.sum() == 0)
        return None

    stalls = 0
    while sum(have.values()) < sum(target.values()):
        left = cfg.time_budget_h * 3600 - (time.time() - t_start)
        if left < 15 * 60:
            _log(cfg, "[final] time budget reached -> stopping with what we have"); break
        need = {c: target[c] - have[c] for c in target}
        rows = generate_step(ctx, cfg, gen, guidance, rng, novelty, rej, need=need, accept=accept)
        with open(path, "a") as fh:
            for x, m, s, y in rows:
                fh.write(_row_json(x, m, s, y) + "\n")
        json.dump(rej, open(_p(cfg, "rejections_final.json"), "w"), indent=2)
        stalls = stalls + 1 if not rows else 0
        if stalls >= 5:
            _log(cfg, "[final] 5 empty batches in a row -> stopping"); break
        _log(cfg, f"[final] {sum(have.values())}/{sum(target.values())}  (+{len(rows)}; "
                  f"{gen.tokens_out / max(gen.seconds, 1e-9):.0f} tok/s)")
    return finalize(ctx, cfg)


def finalize(ctx, cfg):
    recs = [_row_load(l)[0] for l in open(_p(cfg, "synthetic_rows.jsonl"))]
    SX, SM, SS, SY = _stack(recs)
    SM = SM.astype(np.int8); SS = SS.astype(np.int8); SY = SY.astype(np.int8)
    for n, a in [("synth_X_seq", SX), ("synth_mask", SM), ("synth_S_gender", SS), ("synth_y", SY)]:
        np.save(_p(cfg, f"{n}.npy"), a)
    rows = []
    for i in range(len(SX)):
        for w in np.where(SM[i] == 1)[0]:
            r = dict(synth_id=i, week=int(w), S_gender=int(SS[i]), label=int(SY[i]))
            r.update({f: float(SX[i, w, j]) for j, f in enumerate(FEATURES)})
            rows.append(r)
    pd.DataFrame(rows).to_csv(_p(cfg, "synthetic_weekly_long.csv"), index=False)
    # final check: D1 / D2 on the delivered data set
    sv = build_matrix(SX, SM)
    d1, _ = score_D1(ctx, sv, SS, SY, cfg.seed)
    d2, li = score_D2(sv, SS, SY, cfg.seed)
    d = ctx["data"]; tr = ctx["tr"]
    chk = dict(mode=cfg.mode, cohort=cfg.cohort, n=len(SX), D1_acc=d1, D2_auc=d2,
               D2_leak=vec_name(li), real_D2_auc=ctx["real_D2_auc"],
               label_SPD_synth=spd(SY, SS), label_SPD_real=spd(d["y"][tr], d["S"][tr]),
               female_rate=float(SS.mean()), pos_rate=float(SY.mean()),
               pos_rate_F=float(SY[SS == 1].mean()), pos_rate_M=float(SY[SS == 0].mean()),
               weeks=dict(zip(*[a.tolist() for a in np.unique(SM.sum(1), return_counts=True)])),
               all_zero_share={yy: float((SX[SY == yy].sum((1, 2)) == 0).mean()) for yy in (0, 1)},
               qz_active_weeks_pct=float(100 * (SX[:, :, 3][SM.astype(bool)] > 0).mean()),
               sc_active_weeks_pct=float(100 * (SX[:, :, SCORE_IDX][SM.astype(bool)] > 0).mean()),
               real_qz_active_weeks_pct=ctx["real_rates"]["qz"],
               real_sc_active_weeks_pct=ctx["real_rates"]["sc"])
    rf = _p(cfg, "rejections_final.json")
    if os.path.exists(rf):
        rj = json.load(open(rf))
        if rj.get("plan_n"):
            chk["plan_recall_final"] = rj["plan_recall_sum"] / rj["plan_n"]
            chk["plan_mismatch_final"] = rj.get("plan_mismatch", 0)
    save_json(chk, _p(cfg, "final_check.json"))
    _log(cfg, f"[final] saved {len(SX)} students. check: {json.dumps(chk, default=str)}")
    return dict(X=SX, mask=SM, S=SS, y=SY)


# =============================================================================
# MAIN
# =============================================================================
def run(cfg: Config):
    cfg.resolve()
    os.makedirs(cfg.OUT_DIR, exist_ok=True)
    save_json(asdict(cfg), _p(cfg, "config.json"))
    t_start = time.time()
    np.random.seed(cfg.seed)
    ctx = build_context(cfg)
    gen = make_generator(cfg, ctx)
    ctx["system"] = system_prompt(cfg, ctx)
    # output length: tokens of a long real student x students per prompt (+25%)
    d = ctx["data"]
    longest = [serialize(d["X"][i], d["mask"][i], 0, 0, cfg.fmt)
               for i in ctx["tr"][np.argsort(-d["X"][ctx["tr"]].sum((1, 2)))[:20]]]
    per = max(gen.n_tokens(s) for s in longest)
    ctx["max_new_tokens"] = int(per * cfg.students_per_prompt * 1.25) + 64
    _log(cfg, f"[run] v4.3 plan_mode={cfg.plan_mode} mode={cfg.mode} fmt={cfg.fmt} optimizer={cfg.optimizer} activity_plan={cfg.activity_plan} "
              f"max_new_tokens={ctx['max_new_tokens']}")
    _log(cfg, "\n==== ADVERSARIAL PROMPT OPTIMISATION ====")
    st = train_loop(ctx, cfg, gen, t_start)
    _log(cfg, "\n==== FINAL GENERATION ====")
    synth = final_generation(ctx, cfg, gen, st["best_guidance"], t_start)
    _log(cfg, f"[run] done in {(time.time() - t_start) / 3600:.2f} h")
    return synth, st


if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser()
    for k, v in asdict(Config()).items():
        if k in ("fmt", "optimizer"):
            continue
        ap.add_argument(f"--{k}", type=type(v) if not isinstance(v, bool) else
                        (lambda s: s.lower() in ("1", "true", "yes")), default=v)
    run(Config(**vars(ap.parse_args())))


In [ ]:
%%writefile evaluate_v3.py
# -*- coding: utf-8 -*-
"""
evaluate_v3.py  --  evaluation for RQ1, RQ2, RQ3 (FairMALLM-GAN v3)
===================================================================
For every synthetic data set (FairMALLM full, no_fair ablation, MALLM-GAN flat,
CTGAN, old v2 ...) this script computes, on the SAME real test students:

 Utility (RQ2/RQ3) - GRU early-warning model, 5 seeds, mean +/- sd
   TRTR    : train on all real training students
   TRTR-n  : train on n real students (n = synthetic size)  <- fair size reference
   TSTR    : train on the synthetic students
   AUG     : n real + synthetic (data-augmentation view)
   metrics : accuracy, balanced accuracy, F1, ROC-AUC (threshold tuned on the
             validation split of the TRAINING source, never on real test data)
 Fairness of the predictions (RQ1/RQ3) on the real test set:
   SPD = P(yhat=1|S=0) - P(yhat=1|S=1),  EOD = TPR(S=0) - TPR(S=1); per-module SPD
 Data-level fairness (RQ1): label SPD of the data set, D2 AUC (gender from
   features + label) vs. the same quantity on a same-size REAL sample
 Temporal fidelity (RQ2): D1 accuracy (LR, CV), post-hoc GRU discriminative
   score |acc-0.5|, Wasserstein distance per feature (log scale), weekly-mean
   curve error, lag-1 autocorrelation gap, cross-feature correlation gap
 Privacy: distance to closest real training record (DCR) vs. real test -> train

Usage:
  python evaluate_v3.py --data_dir <oulad_processed> --out_dir <folder> \
     --synth full=<dir> no_fair=<dir> mallm_flat=<dir> ctgan=<dir> [--cohort active]
Outputs: results_v3.json, utility_fairness.csv, data_quality.csv, figures/*.png, report.md
"""
import argparse, os, json, time
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import NearestNeighbors
from scipy.stats import wasserstein_distance

from common import (FEATURES, load_real, get_split, build_matrix, spd, eod, flat_log,
                    save_json, vec_name)

DEV = "cuda" if torch.cuda.is_available() else "cpu"


# =============================================================================
# GRU early-warning model
# =============================================================================
class GRUClassifier(nn.Module):
    def __init__(self, n_feat, hidden=64):
        super().__init__()
        self.gru = nn.GRU(n_feat, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1))

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True,
                                                   enforce_sorted=False)
        _, h = self.gru(packed)
        return self.head(h[-1]).squeeze(-1)


class Scaler:
    """log1p, then standardise with REAL training weeks only; padded weeks -> 0."""
    def fit(self, X, M):
        v = np.log1p(np.clip(X, 0, None))[M.astype(bool)]
        self.mu, self.sd = v.mean(0), v.std(0) + 1e-6
        return self

    def __call__(self, X, M):
        Z = (np.log1p(np.clip(X, 0, None)) - self.mu) / self.sd
        Z[M == 0] = 0
        return Z.astype(np.float32)


def _tensors(Z, M):
    L = np.clip(M.sum(1), 1, Z.shape[1]).astype(np.int64)
    return torch.tensor(Z), torch.tensor(L)


@torch.no_grad()
def predict_proba(model, Z, M, bs=4096):
    model.eval(); out = []
    x, l = _tensors(Z, M)
    for i in range(0, len(x), bs):
        out.append(torch.sigmoid(model(x[i:i + bs].to(DEV), l[i:i + bs])).cpu().numpy())
    return np.concatenate(out)


def train_gru(Ztr, Mtr, ytr, Zva, Mva, yva, seed, epochs=40, patience=5, lr=1e-3, bs=128):
    """Adam + BCE, early stopping on validation ROC-AUC of the TRAINING SOURCE
    (real val for TRTR, held-out synthetic for TSTR -> no real-data leakage)."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = GRUClassifier(Ztr.shape[2]).to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss()
    x, l = _tensors(Ztr, Mtr); yt = torch.tensor(ytr.astype(np.float32))
    best, best_state, bad = -1, None, 0
    g = torch.Generator().manual_seed(seed)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(x), generator=g)
        for i in range(0, len(x), bs):
            b = perm[i:i + bs]
            opt.zero_grad()
            loss = lossf(model(x[b].to(DEV), l[b]), yt[b].to(DEV))
            loss.backward(); opt.step()
        p = predict_proba(model, Zva, Mva)
        auc = roc_auc_score(yva, p) if len(np.unique(yva)) > 1 else 0.5
        if auc > best + 1e-4:
            best, bad = auc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    # decision threshold chosen on the SAME validation data (max. balanced accuracy);
    # a fixed 0.5 collapses to "all negative" for weak small-data models
    p = predict_proba(model, Zva, Mva)
    grid = np.unique(np.quantile(p, np.linspace(0.02, 0.98, 49)))
    model.threshold = float(max(grid, key=lambda t: balanced_accuracy_score(yva, p >= t))) \
        if len(np.unique(yva)) > 1 else 0.5
    return model


def metrics(y, p, S, modules=None, thr=0.5):
    yh = (p >= thr).astype(int)
    r = dict(acc=accuracy_score(y, yh), bal_acc=balanced_accuracy_score(y, yh),
             f1=f1_score(y, yh, zero_division=0), auc=roc_auc_score(y, p),
             SPD=spd(yh, S), EOD=eod(y, yh, S), pos_pred_rate=float(yh.mean()))
    if modules is not None:
        r["SPD_by_module"] = {m: spd(yh[modules == m], S[modules == m])
                              for m in np.unique(modules)
                              if len(np.unique(S[modules == m])) == 2}
    return r


def summarise(runs):
    keys = [k for k in runs[0] if k != "SPD_by_module"]
    out = {k: float(np.mean([r[k] for r in runs])) for k in keys}
    out.update({k + "_sd": float(np.std([r[k] for r in runs])) for k in keys})
    if "SPD_by_module" in runs[0]:
        mods = runs[0]["SPD_by_module"].keys()
        out["SPD_by_module"] = {m: float(np.mean([r["SPD_by_module"][m] for r in runs])) for m in mods}
    out["n_runs"] = len(runs)
    return out


def split_val(n, y, seed, frac=0.15):
    idx = np.arange(n)
    tr, va = train_test_split(idx, test_size=frac, random_state=seed, stratify=y)
    return tr, va


def run_condition(name, X, M, y, sc, test, seeds, log, sample_fn=None):
    """sample_fn(seed) -> (X, M, y) lets TRTR-n draw a new real subsample per seed."""
    Zte, Mte, yte, Ste, mod = test
    runs = []
    for sd in seeds:
        Xs, Ms, ys = sample_fn(sd) if sample_fn else (X, M, y)
        a, b = split_val(len(ys), ys, sd)
        mdl = train_gru(sc(Xs[a], Ms[a]), Ms[a], ys[a], sc(Xs[b], Ms[b]), Ms[b], ys[b], sd)
        runs.append(metrics(yte, predict_proba(mdl, Zte, Mte), Ste, mod, mdl.threshold))
    s = summarise(runs)
    log(f"  {name:28s} acc={s['acc']:.3f}±{s['acc_sd']:.3f} bal={s['bal_acc']:.3f} "
        f"f1={s['f1']:.3f} auc={s['auc']:.3f}±{s['auc_sd']:.3f} SPD={s['SPD']:+.3f} EOD={s['EOD']:+.3f}")
    return s


# =============================================================================
# Data-level quality: fidelity, fairness, privacy
# =============================================================================
def _clf():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))


def d1_acc(rv, sv, seed=0):
    X = np.vstack([rv, sv]); lab = np.r_[np.ones(len(rv)), np.zeros(len(sv))]
    p = cross_val_predict(_clf(), X, lab, cv=StratifiedKFold(5, shuffle=True, random_state=seed))
    return float(accuracy_score(lab, p))


def d2_auc(v, S, y, seed=0):
    X = np.column_stack([v, y])
    p = cross_val_predict(_clf(), X, S, cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                          method="predict_proba")[:, 1]
    return float(roc_auc_score(S, p))


def matched_real(d, tr, S, y, rng):
    idx = []
    for s in (0, 1):
        for yy in (0, 1):
            k = int(((S == s) & (y == yy)).sum())
            pool = tr[(d["S"][tr] == s) & (d["y"][tr] == yy)]
            if k:
                idx += list(rng.choice(pool, k, replace=len(pool) < k))
    return np.array(idx)


def discriminative_score(Xr, Mr, Xs, Ms, sc, seed=0):
    """Post-hoc GRU discriminator (TimeGAN-style). Returns |acc - 0.5| on held-out data."""
    n = min(len(Xr), len(Xs))
    X = np.concatenate([Xr[:n], Xs[:n]]); M = np.concatenate([Mr[:n], Ms[:n]])
    lab = np.r_[np.ones(n), np.zeros(n)]
    a, b = train_test_split(np.arange(2 * n), test_size=0.3, random_state=seed, stratify=lab)
    a2, v = train_test_split(a, test_size=0.2, random_state=seed, stratify=lab[a])
    mdl = train_gru(sc(X[a2], M[a2]), M[a2], lab[a2], sc(X[v], M[v]), M[v], lab[v], seed, epochs=20)
    p = predict_proba(mdl, sc(X[b], M[b]), M[b])
    return float(abs(accuracy_score(lab[b], p >= mdl.threshold) - 0.5))


def fidelity(Xr, Mr, Xs, Ms):
    out = {}
    br, bs = Mr.astype(bool), Ms.astype(bool)
    for f, nm in enumerate(FEATURES):
        vr, vs = np.log1p(Xr[:, :, f][br]), np.log1p(Xs[:, :, f][bs])
        out[f"W1_{nm}"] = float(wasserstein_distance(vr, vs))
        cr = [np.log1p(Xr[:, w, f][br[:, w]]).mean() for w in range(Xr.shape[1]) if br[:, w].any() and bs[:, w].any()]
        cs = [np.log1p(Xs[:, w, f][bs[:, w]]).mean() for w in range(Xs.shape[1]) if br[:, w].any() and bs[:, w].any()]
        out[f"curveMAE_{nm}"] = float(np.mean(np.abs(np.array(cr) - np.array(cs))))
        out[f"lag1_gap_{nm}"] = float(abs(_lag1(Xr[:, :, f], Mr) - _lag1(Xs[:, :, f], Ms)))
    out["W1_mean"] = float(np.mean([out[f"W1_{n}"] for n in FEATURES]))
    out["curveMAE_mean"] = float(np.mean([out[f"curveMAE_{n}"] for n in FEATURES]))
    out["lag1_gap_mean"] = float(np.mean([out[f"lag1_gap_{n}"] for n in FEATURES]))
    cr = np.corrcoef(np.log1p(Xr[br]).T); cs = np.corrcoef(np.log1p(Xs[bs]).T)
    out["corr_gap_fro"] = float(np.nan_to_num(np.linalg.norm(cr - cs)))
    return out


def _lag1(V, M):
    V = np.log1p(V); a, b = [], []
    for w in range(V.shape[1] - 1):
        k = (M[:, w] == 1) & (M[:, w + 1] == 1)
        a.append(V[k, w]); b.append(V[k, w + 1])
    a, b = np.concatenate(a), np.concatenate(b)
    return float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else 0.0


def privacy(d, tr, te, Xs, Ms):
    rf = flat_log(d["X"][tr], d["mask"][tr])
    nn_ = NearestNeighbors(n_neighbors=1).fit(rf)
    sf = flat_log(Xs, Ms); tf = flat_log(d["X"][te], d["mask"][te])
    rich_s, rich_t = (sf > 0).sum(1) >= 5, (tf > 0).sum(1) >= 5
    ds = nn_.kneighbors(sf[rich_s])[0][:, 0] if rich_s.any() else np.array([np.nan])
    dt = nn_.kneighbors(tf[rich_t])[0][:, 0]
    return dict(DCR_synth_median=float(np.nanmedian(ds)), DCR_synth_p05=float(np.nanpercentile(ds, 5)),
                DCR_realtest_median=float(np.median(dt)), DCR_realtest_p05=float(np.percentile(dt, 5)),
                exact_copies=int((ds == 0).sum()),
                share_closer_than_real_p05=float((ds < np.percentile(dt, 5)).mean()))


def data_quality(name, d, tr, te, Xs, Ms, Ss, ys, sc, rng, log):
    q = dict(n=len(Xs), female_rate=float(Ss.mean()), pos_rate=float(ys.mean()),
             label_SPD=spd(ys, Ss), all_zero_share=float((Xs.sum((1, 2)) == 0).mean()))
    m = matched_real(d, tr, Ss, ys, rng)
    rv, sv = build_matrix(d["X"][m], d["mask"][m]), build_matrix(Xs, Ms)
    q["D1_acc"] = d1_acc(rv, sv)
    q["D2_auc"] = d2_auc(sv, Ss, ys) if len(np.unique(Ss)) == 2 else float("nan")
    q["D2_auc_real_same_size"] = d2_auc(rv, d["S"][m], d["y"][m])
    q["disc_score_GRU"] = discriminative_score(d["X"][m], d["mask"][m], Xs, Ms, sc)
    q.update(fidelity(d["X"][m], d["mask"][m], Xs, Ms))
    q.update(privacy(d, tr, te, Xs, Ms))
    log(f"  [{name}] n={q['n']} D1={q['D1_acc']:.3f} D2={q['D2_auc']:.3f} (real {q['D2_auc_real_same_size']:.3f}) "
        f"labelSPD={q['label_SPD']:+.3f} disc={q['disc_score_GRU']:.3f} W1={q['W1_mean']:.3f} "
        f"DCR med={q['DCR_synth_median']:.2f} (real {q['DCR_realtest_median']:.2f})")
    return q


# =============================================================================
# Figures + report
# =============================================================================
def figures(out_dir, d, tr, synths, uf):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(os.path.join(out_dir, "figures"), exist_ok=True)
    fig, ax = plt.subplots(1, 5, figsize=(22, 4))
    Xr, Mr = d["X"][tr], d["mask"][tr].astype(bool)
    for f, nm in enumerate(FEATURES):
        ax[f].plot([np.log1p(Xr[:, w, f][Mr[:, w]]).mean() for w in range(12)], "k-o", lw=2, label="real")
        for name, (Xs, Ms, _, _) in synths.items():
            Mb = Ms.astype(bool)
            ax[f].plot([np.log1p(Xs[:, w, f][Mb[:, w]]).mean() if Mb[:, w].any() else np.nan
                        for w in range(12)], "-o", ms=3, label=name)
        ax[f].set_title(nm); ax[f].set_xlabel("week"); ax[f].set_ylabel("mean log1p(value)")
    ax[0].legend(fontsize=8); fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "figures", "weekly_curves.png"), dpi=130); plt.close(fig)
    # fairness-utility trade-off (RQ3)
    fig, ax = plt.subplots(figsize=(6.5, 5))
    for _, r in uf.iterrows():
        ax.errorbar(abs(r["SPD"]), r["auc"], yerr=r["auc_sd"], fmt="o")
        ax.annotate(r["condition"], (abs(r["SPD"]), r["auc"]), fontsize=7, xytext=(4, 3),
                    textcoords="offset points")
    ax.set_xlabel("|SPD| of predictions on real test set (lower = fairer)")
    ax.set_ylabel("ROC-AUC on real test set (higher = more useful)")
    ax.set_title("Fairness-utility trade-off"); ax.grid(alpha=.3); fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "figures", "fairness_utility.png"), dpi=130); plt.close(fig)


def write_report(out_dir, uf, dq, cohort, notes):
    L = [f"# Evaluation report (cohort = {cohort})", "", *notes, "",
         "## Utility and fairness of the GRU on the REAL test set (mean over seeds)", "",
         "| condition | n train | acc | bal. acc | F1 | AUC | SPD | EOD |", "|---|---|---|---|---|---|---|---|"]
    for _, r in uf.iterrows():
        L.append(f"| {r['condition']} | {int(r['n_train'])} | {r['acc']:.3f}±{r['acc_sd']:.3f} | "
                 f"{r['bal_acc']:.3f} | {r['f1']:.3f} | {r['auc']:.3f}±{r['auc_sd']:.3f} | "
                 f"{r['SPD']:+.3f} | {r['EOD']:+.3f} |")
    if len(dq):
        L += ["", "## Synthetic data quality", "",
              "| data set | n | label SPD | D1 acc (→0.5) | D2 AUC (→0.5) | D2 real same n | GRU disc. (→0) "
              "| W1 mean | lag-1 gap | DCR median (real test) |", "|---|---|---|---|---|---|---|---|---|---|"]
        for _, r in dq.iterrows():
            L.append(f"| {r['dataset']} | {int(r['n'])} | {r['label_SPD']:+.3f} | {r['D1_acc']:.3f} | "
                     f"{r['D2_auc']:.3f} | {r['D2_auc_real_same_size']:.3f} | {r['disc_score_GRU']:.3f} | "
                     f"{r['W1_mean']:.3f} | {r['lag1_gap_mean']:.3f} | {r['DCR_synth_median']:.2f} "
                     f"({r['DCR_realtest_median']:.2f}) |")
    L += ["", "Figures: `figures/weekly_curves.png`, `figures/fairness_utility.png`."]
    open(os.path.join(out_dir, "report.md"), "w").write("\n".join(L))


# =============================================================================
# MAIN
# =============================================================================
def run(data_dir, out_dir, synth_dirs, cohort="active", seeds=(0, 1, 2, 3, 4), augment=True,
        quality=True, seed=42):
    os.makedirs(out_dir, exist_ok=True)
    logf = open(os.path.join(out_dir, "eval_log.txt"), "a")
    def log(m): print(m, flush=True); logf.write(m + "\n"); logf.flush()
    t0 = time.time()
    d = load_real(data_dir)
    tr, va, te = get_split(d, cohort, seed)
    X, M, y, S = d["X"], d["mask"], d["y"], d["S"]
    mod = d["static"]["code_module"].values
    sc = Scaler().fit(X[tr], M[tr])
    test = (sc(X[te], M[te]), M[te], y[te], S[te], mod[te])
    log(f"[eval] cohort={cohort} train={len(tr)} val={len(va)} test={len(te)} device={DEV}")
    synths = {}
    for name, p in synth_dirs.items():
        try:
            synths[name] = (np.load(f"{p}/synth_X_seq.npy").astype(np.float32),
                            np.load(f"{p}/synth_mask.npy").astype(np.int8),
                            np.load(f"{p}/synth_S_gender.npy").astype(np.int8),
                            np.load(f"{p}/synth_y.npy").astype(np.int8))
            log(f"[eval] loaded {name}: {len(synths[name][0])} students from {p}")
        except FileNotFoundError:
            log(f"[eval] WARNING: no synthetic files in {p} -> skipped")
    res, rows = {}, []

    def add(cond, n, s):
        res[cond] = s; rows.append(dict(condition=cond, n_train=n, **{k: v for k, v in s.items()
                                                                       if k != "SPD_by_module"}))
    log("[eval] utility + fairness (GRU, real test set)")
    # TRTR: real train, early stopping on real val
    runs = []
    for sd in seeds:
        mdl = train_gru(sc(X[tr], M[tr]), M[tr], y[tr], sc(X[va], M[va]), M[va], y[va], sd)
        runs.append(metrics(y[te], predict_proba(mdl, *test[:2]), S[te], mod[te], mdl.threshold))
    s = summarise(runs); add("TRTR (all real)", len(tr), s)
    log(f"  {'TRTR (all real)':28s} acc={s['acc']:.3f}±{s['acc_sd']:.3f} auc={s['auc']:.3f} "
        f"SPD={s['SPD']:+.3f} EOD={s['EOD']:+.3f}")
    sizes = sorted({len(v[0]) for v in synths.values()})
    for n in sizes:
        def real_n(sd, n=n):
            ix = train_test_split(tr, train_size=min(n, len(tr) - 2), random_state=1000 + sd,
                                  stratify=y[tr] * 2 + S[tr])[0]
            return X[ix], M[ix], y[ix]
        add(f"TRTR-n (real n={n})", n, run_condition(f"TRTR-n (real n={n})", None, None, None, sc,
                                                       test, seeds, log, real_n))
    for name, (Xs, Ms, Ss, ys) in synths.items():
        add(f"TSTR {name}", len(Xs), run_condition(f"TSTR {name}", Xs, Ms, ys, sc, test, seeds, log))
        if augment:
            def aug(sd, Xs=Xs, Ms=Ms, ys=ys):
                n = len(Xs)
                ix = train_test_split(tr, train_size=min(n, len(tr) - 2), random_state=1000 + sd,
                                      stratify=y[tr] * 2 + S[tr])[0]
                return (np.concatenate([X[ix], Xs]), np.concatenate([M[ix], Ms]),
                        np.concatenate([y[ix], ys]))
            add(f"AUG real-n + {name}", 2 * len(Xs), run_condition(f"AUG real-n + {name}", None, None,
                                                                    None, sc, test, seeds, log, aug))
    uf = pd.DataFrame(rows)
    uf.to_csv(os.path.join(out_dir, "utility_fairness.csv"), index=False)

    dq = pd.DataFrame()
    if quality and synths:
        log("[eval] synthetic data quality (fidelity / fairness / privacy)")
        rng = np.random.default_rng(seed)
        q = {name: data_quality(name, d, tr, te, *v, sc, rng, log) for name, v in synths.items()}
        res["data_quality"] = q
        dq = pd.DataFrame([dict(dataset=k, **v) for k, v in q.items()])
        dq.to_csv(os.path.join(out_dir, "data_quality.csv"), index=False)
    res["real_reference"] = dict(label_SPD_train=spd(y[tr], S[tr]), pos_rate_train=float(y[tr].mean()),
                                 female_rate_train=float(S[tr].mean()))
    save_json(res, os.path.join(out_dir, "results_v3.json"))
    figures(out_dir, d, tr, synths, uf)
    notes = [f"Real training data: n={len(tr)}, positive rate {y[tr].mean():.3f}, "
             f"label SPD {spd(y[tr], S[tr]):+.3f} (S=1 female).",
             "SPD = P(ŷ=1|S=0) − P(ŷ=1|S=1); EOD = TPR(S=0) − TPR(S=1). Target for both: 0.",
             "TRTR-n uses the same number of real students as the synthetic set (fair size reference)."]
    write_report(out_dir, uf, dq, cohort, notes)
    log(f"[eval] done in {(time.time() - t0) / 60:.1f} min -> {out_dir}")
    return res


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--synth", nargs="*", default=[], help="name=dir pairs")
    ap.add_argument("--cohort", default="active", choices=["active", "all"])
    ap.add_argument("--seeds", type=int, default=5)
    ap.add_argument("--no_augment", action="store_true")
    ap.add_argument("--no_quality", action="store_true")
    a = ap.parse_args()
    run(a.data_dir, a.out_dir, dict(s.split("=", 1) for s in a.synth), a.cohort,
        tuple(range(a.seeds)), not a.no_augment, not a.no_quality)


In [ ]:
import glob, os, sys, json, torch
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
hits = glob.glob("/kaggle/input/**/oulad_processed/X_seq.npy", recursive=True) or \
       glob.glob("/kaggle/input/**/X_seq.npy", recursive=True)
assert hits, "Attach the dataset 'oulad-processed' (Add Input) first."
DATA_DIR = os.path.dirname(hits[0])
print("DATA_DIR =", DATA_DIR)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, "No GPU: Settings -> Accelerator -> GPU T4 x2"


In [ ]:
MODE = "no_fair"
from fair_mallm_gan_v4 import Config, run
cfg = Config(DATA_DIR=DATA_DIR, OUT_DIR=f"/kaggle/working/run_{MODE}", mode=MODE,
             cohort="active", n_synthetic=2000, max_rounds=10,
             prompts_per_batch=8, students_per_prompt=8, time_budget_h=10.0)
synth, state = run(cfg)
print(open(f"{cfg.OUT_DIR}/final_check.json").read())


In [ ]:
from evaluate_v3 import run as run_eval
res = run_eval(DATA_DIR, f"/kaggle/working/eval_{MODE}", {MODE: cfg.OUT_DIR},
               cohort="active", seeds=(0, 1, 2))
print(open(f"/kaggle/working/eval_{MODE}/report.md").read())
